In [ ]:
import numpy as np
from pathlib import Path
import os
from tqdm import tqdm  # 進捗バー表示用（なければ標準出力だけでも可）

# --- 設定 ---
base_dir = Path("/content/group5/Fase2/1x/data/train_v2.0")
videos_dir = base_dir / "videos"
metadata_dir = base_dir / "metadata"

# 集計用変数
global_max_val = 0
file_stats = []
missing_files = []

print(f"Starting analysis for video_0.bin to video_99.bin in {videos_dir}...\n")

# 0から99までループ
for i in tqdm(range(100), desc="Processing files"):
    video_filename = f"video_{i}.bin"
    video_path = videos_dir / video_filename

    # ファイルが存在するか確認
    if not video_path.exists():
        missing_files.append(video_filename)
        continue

    # ファイルサイズからトークン総数を計算 (uint32 = 4 bytes)
    try:
        file_size_bytes = os.path.getsize(video_path)
        total_tokens = file_size_bytes // 4

        # トークン数が0の場合はスキップ
        if total_tokens == 0:
            continue

        # メモリマップで開く
        data = np.memmap(video_path, dtype=np.uint32, mode="r", shape=(total_tokens,))

        # ファイル内の最大値を取得
        # ※ファイル全体を走査するため、ファイルサイズによっては少し時間がかかります
        local_max = data.max()

        # 全体の最大値を更新
        if local_max > global_max_val:
            global_max_val = local_max

        file_stats.append(local_max)

    except Exception as e:
        print(f"\nError processing {video_filename}: {e}")

# --- 集計結果の表示 ---
print("\n" + "="*30)
print("       集計結果")
print("="*30)

if file_stats:
    print(f"検証ファイル数: {len(file_stats)} / 100")
    if missing_files:
        print(f"見つからなかったファイル: {len(missing_files)}個")

    print(f"\n🔹 全体の中での最大トークンID: {global_max_val}")
    print(f"🔹 各ファイルの最大値の平均: {sum(file_stats) / len(file_stats):.2f}")

    print("-" * 30)

    # 結論の判定
    if global_max_val < 64000:
        print(f"✅ 全データを確認しましたが、最大値は {global_max_val} でした。")
        print("   -> Vocab Size は 64,000 (論文準拠) で間違いありません。")
        print("   -> コードの vocab_size 設定は 64000 にしてください。")
    else:
        print(f"⚠️ 最大値 {global_max_val} が見つかりました（64,000超）。")
        print("   -> 論文の記述(64k)とは異なるTokenizerが使われているか、データ生成設定が異なります。")
        print("   -> コードの vocab_size 設定は 262144（またはそれ以上）にする必要があります。")
        print("   -> OOM回避のため、チャンク処理（Memory Efficient Implementation）の実装が必須です。")
else:
    print("有効なデータが見つかりませんでした。パスを確認してください。")

Starting analysis for video_0.bin to video_99.bin in /content/group5/Fase2/1x/data/train_v2.0/videos...



Processing files: 100%|██████████| 100/100 [00:39<00:00,  2.50it/s]


       集計結果
検証ファイル数: 100 / 100

🔹 全体の中での最大トークンID: 63999
🔹 各ファイルの最大値の平均: 63998.29
------------------------------
✅ 全データを確認しましたが、最大値は 63999 でした。
   -> Vocab Size は 64,000 (論文準拠) で間違いありません。
   -> コードの vocab_size 設定は 64000 にしてください。
